## 1. Objective

In this section, fraud detection is approached from an anomaly detection perspective rather than as a standard supervised classification problem.

Given the extreme class imbalance and the rarity of fraudulent transactions, the goal is to identify observations that deviate from typical transactional behaviour, without relying explicitly on fraud labels during model training.

The objective is to evaluate whether unsupervised and semi-supervised methods can effectively isolate fraudulent patterns based on behavioural features derived in previous stages.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/processed/feature_engineered.csv")

df.head()

In [ ]:
X = df.drop(columns=["isFraud"])
y = df["isFraud"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Isolation Forest

Isolation Forest is based on the idea that anomalies are easier to isolate than normal observations.

It works by randomly partitioning the feature space and measuring how quickly observations become isolated. Points that require fewer splits are considered anomalous.

In [ ]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    n_estimators=100,
    contamination=0.001,  # roughly fraud rate
    random_state=42,
    n_jobs=-1
)

iso.fit(X_train)

In [ ]:
y_pred_iso = iso.predict(X_test)

# Convert to binary (1 = fraud)
y_pred_iso = (y_pred_iso == -1).astype(int)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score

print("ROC-AUC:", roc_auc_score(y_test, y_pred_iso))
print("PR-AUC:", average_precision_score(y_test, y_pred_iso))

print(classification_report(y_test, y_pred_iso))

## One-Class SVM

One-Class SVM attempts to learn the boundary of normal behaviour and classifies points outside this boundary as anomalies.

It is more sensitive to scaling and high dimensionality compared to tree-based methods.

In [ ]:
from sklearn.svm import OneClassSVM

svm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.001
)

svm.fit(X_train)

In [ ]:
y_pred_svm = svm.predict(X_test)
y_pred_svm = (y_pred_svm == -1).astype(int)

## Local Outlier Factor

LOF measures the local density deviation of a given data point with respect to its neighbours.

Points with significantly lower density are considered anomalies.

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=0.001
)

y_pred_lof = lof.fit_predict(X_test)
y_pred_lof = (y_pred_lof == -1).astype(int)

## Autoencoder

Autoencoders learn to reconstruct normal patterns. Transactions that are poorly reconstructed are considered anomalous.

This approach captures non-linear relationships and is well suited for behavioural anomaly detection.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

input_dim = X_train.shape[1]

model = models.Sequential([
    layers.Dense(16, activation="relu", input_shape=(input_dim,)),
    layers.Dense(8, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(input_dim, activation="linear")
])

model.compile(optimizer="adam", loss="mse")

model.fit(X_train, X_train, epochs=10, batch_size=256, validation_split=0.1)

In [ ]:
recon = model.predict(X_test)
mse = ((X_test - recon) ** 2).mean(axis=1)

threshold = np.percentile(mse, 99.9)

y_pred_ae = (mse > threshold).astype(int)

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Isolation Forest", "One-Class SVM", "LOF", "Autoencoder"],
    "PR-AUC": [
        average_precision_score(y_test, y_pred_iso),
        average_precision_score(y_test, y_pred_svm),
        average_precision_score(y_test, y_pred_lof),
        average_precision_score(y_test, y_pred_ae),
    ]
})

comparison.sort_values(by="PR-AUC", ascending=False)